# SVMModelEva

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from evaluation_utils import (
    load_yelp_split,
    print_evaluation,
    print_top_terms,
    validate_pipeline_labels,
)


PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
BASELINE_MODEL_FILE = PART_B_DIR / "models" / "svm_pipeline.joblib"
MODEL_FILE = PART_B_DIR / "models" / "svm_tuned.joblib"

## Load Tuned Model

In [2]:
# Load the tuned Yelp Linear SVM pipeline saved by Q3.
pipeline = joblib.load(MODEL_FILE)
validate_pipeline_labels(pipeline, "Linear SVM")

## Model Evaluation

In [3]:
# Recreate the same held-out test set used throughout the SVM workflow.
df, X_train, X_test, y_train, y_test = load_yelp_split(DATA_FILE)

print_top_terms(pipeline, X_train, y_train, "Linear SVM")
print_evaluation(pipeline, X_test, y_test, "Linear SVM")

=== Top Terms Per Sentiment Class (Linear SVM) ===

negative:
two star, not, terrible, not worth, bad, not good, bland, horrible, not back, waste

neutral:
three star, hit miss, not bad, not amazing, decent, okay, not best, little disappointed, though, convenient

positive:
amaze, great, delicious, best, not disappoint, awesome, perfect, excellent, go wrong, fantastic

=== Q4: Tuned Yelp Linear SVM 3-Class Classification Report ===
              precision    recall  f1-score   support

    negative     0.8196    0.8308    0.8252      2400
     neutral     0.4860    0.4492    0.4669      1200
    positive     0.8169    0.8367    0.8267      2400

    accuracy                         0.7568      6000
   macro avg     0.7075    0.7056    0.7062      6000
weighted avg     0.7518    0.7568    0.7541      6000

=== Q4: Tuned Yelp Linear SVM Summary ===
Accuracy:        0.7568
Macro Precision: 0.7075
Macro Recall:    0.7056
Macro F1-score:  0.7062
Weighted F1:     0.7541

=== Q4: Tuned Yelp L

## Before vs After Tuning

The Q2 baseline and Q4 tuned SVM are evaluated on the same held-out test set. The table shows whether tuning improved each main metric.

In [ ]:
# Compare the Q2 baseline and tuned SVM on the identical held-out test set.
baseline_pipeline = joblib.load(BASELINE_MODEL_FILE)
validate_pipeline_labels(baseline_pipeline, "Baseline Linear SVM")


def summarize_model(model):
    predictions = model.predict(X_test)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_test, predictions, average="macro", zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        y_test, predictions, average="weighted", zero_division=0
    )[2]
    scores = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
    }
    return predictions, scores


baseline_predictions, baseline_scores = summarize_model(baseline_pipeline)
tuned_predictions, tuned_scores = summarize_model(pipeline)
comparison = pd.DataFrame(
    {"Q2 Baseline": baseline_scores, "Q4 Tuned": tuned_scores}
)
comparison["Improvement"] = comparison["Q4 Tuned"] - comparison["Q2 Baseline"]

print("=== Linear SVM: Before vs After Tuning ===")
print(
    comparison.to_string(
        formatters={
            "Q2 Baseline": lambda value: f"{value:.4f}",
            "Q4 Tuned": lambda value: f"{value:.4f}",
            "Improvement": lambda value: f"{value:+.4f}",
        }
    )
)

baseline_correct = int((baseline_predictions == y_test).sum())
tuned_correct = int((tuned_predictions == y_test).sum())
baseline_errors = len(y_test) - baseline_correct
tuned_errors = len(y_test) - tuned_correct

print(f"\nAdditional correct predictions: {tuned_correct - baseline_correct:+d}")
print(f"Errors: {baseline_errors} -> {tuned_errors}")
print(
    "Relative error reduction: "
    f"{(baseline_errors - tuned_errors) / baseline_errors * 100:.2f}%"
)

=== Linear SVM: Before vs After Tuning ===
                Q2 Baseline Q4 Tuned Improvement
Accuracy             0.7455   0.7568     +0.0113
Macro Precision      0.6968   0.7075     +0.0107
Macro Recall         0.6961   0.7056     +0.0094
Macro F1             0.6964   0.7062     +0.0098
Weighted F1          0.7443   0.7541     +0.0098

Additional correct predictions: +68
Errors: 1527 -> 1459
Relative error reduction: 4.45%
